# 🧠 Customer Churn Prediction — Neural Network with TensorFlow/Keras

> **Author:** Sourav Nayak | [LinkedIn](https://www.linkedin.com/in/sourav-nayak-756690306) | [GitHub](https://github.com/SOURAV033)
>
> **Tech Stack:** Python · TensorFlow 2.x · Keras · Pandas · NumPy · scikit-learn · Matplotlib · Seaborn

---

## 📌 Project Overview

This notebook extends the [Customer Churn Prediction ML project](https://github.com/SOURAV033/customer-churn-prediction) by implementing a **Deep Neural Network (DNN)** using TensorFlow/Keras — comparing its performance against the classical ML models already built.

### What this notebook covers:
| Step | Description |
|------|-------------|
| 1 | Data loading, cleaning & feature engineering |
| 2 | Preprocessing — encoding, scaling, train/test split |
| 3 | Neural Network architecture design |
| 4 | Model training with callbacks (EarlyStopping, ReduceLROnPlateau) |
| 5 | Training history visualisation (loss & accuracy curves) |
| 6 | Model evaluation — ROC-AUC, F1, Confusion Matrix |
| 7 | Comparison: Neural Network vs Classical ML |
| 8 | Single customer churn prediction demo |

### Neural Network Architecture
```
Input Layer  →  24 features
    ↓
Dense(64)    →  ReLU  →  BatchNorm  →  Dropout(0.3)
    ↓
Dense(32)    →  ReLU  →  BatchNorm  →  Dropout(0.2)
    ↓
Dense(16)    →  ReLU  →  Dropout(0.1)
    ↓
Dense(1)     →  Sigmoid  →  Churn Probability
```

## 1. Install & Import Libraries

In [ ]:
# Run this cell first if using Google Colab
# !pip install tensorflow pandas numpy scikit-learn matplotlib seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization, Input
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

# scikit-learn utilities
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score,
    accuracy_score, precision_score, recall_score
)

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#F8F9FA',
    'axes.facecolor':   '#F8F9FA',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'DejaVu Sans'
})

print(f'TensorFlow version : {tf.__version__}')
print(f'Keras version      : {keras.__version__}')
print(f'NumPy version      : {np.__version__}')
print(f'Pandas version     : {pd.__version__}')
print('\n✅ All libraries loaded successfully!')

## 2. Load & Inspect Data

In [ ]:
# Load dataset (same as the main churn project)
# If running locally, use: df = pd.read_csv('../data/telco_churn.csv')
# If running on Colab, upload the CSV or use the URL below

# Option A — Local
# df = pd.read_csv('../data/telco_churn.csv')

# Option B — Generate synthetic dataset (works anywhere, no file needed)
np.random.seed(42)
n = 7043

tenure      = np.random.randint(0, 73, n)
monthly     = np.round(np.random.uniform(18, 120, n), 2)
total       = np.clip(np.round(monthly * tenure + np.random.normal(0, 50, n), 2), 0, None)
contracts   = np.random.choice(['Month-to-month','One year','Two year'], n, p=[0.55,0.25,0.20])
internet    = np.random.choice(['DSL','Fiber optic','No'], n, p=[0.34,0.44,0.22])
payment     = np.random.choice(['Electronic check','Mailed check','Bank transfer','Credit card'], n, p=[0.34,0.23,0.22,0.21])
gender      = np.random.choice(['Male','Female'], n)
partner     = np.random.choice(['Yes','No'], n, p=[0.48,0.52])
dependents  = np.random.choice(['Yes','No'], n, p=[0.30,0.70])
phone       = np.random.choice(['Yes','No'], n, p=[0.90,0.10])
multi_lines = np.where(phone=='No','No phone service', np.random.choice(['Yes','No'],n,p=[0.53,0.47]))
online_sec  = np.where(internet=='No','No internet service', np.random.choice(['Yes','No'],n,p=[0.29,0.71]))
online_bk   = np.where(internet=='No','No internet service', np.random.choice(['Yes','No'],n,p=[0.28,0.72]))
device_prot = np.where(internet=='No','No internet service', np.random.choice(['Yes','No'],n,p=[0.34,0.66]))
tech_supp   = np.where(internet=='No','No internet service', np.random.choice(['Yes','No'],n,p=[0.29,0.71]))
streaming_tv= np.where(internet=='No','No internet service', np.random.choice(['Yes','No'],n,p=[0.38,0.62]))
streaming_m = np.where(internet=='No','No internet service', np.random.choice(['Yes','No'],n,p=[0.39,0.61]))
paperless   = np.random.choice(['Yes','No'], n, p=[0.59,0.41])
senior      = np.random.choice([0,1], n, p=[0.84,0.16])

churn_prob  = (
    0.35*(tenure<12).astype(float) + 0.20*(contracts=='Month-to-month').astype(float) +
    0.15*(internet=='Fiber optic').astype(float) + 0.10*(payment=='Electronic check').astype(float) +
    0.10*(monthly>80).astype(float) + 0.05*(online_sec=='No').astype(float) +
    0.05*(tech_supp=='No').astype(float)
)
churn_prob  = churn_prob / churn_prob.max() * 0.70
churn       = np.where(np.random.uniform(0,1,n) < churn_prob, 'Yes', 'No')

df = pd.DataFrame({
    'customerID': ['CUST-'+str(10000+i) for i in range(n)],
    'gender':gender, 'SeniorCitizen':senior, 'Partner':partner,
    'Dependents':dependents, 'tenure':tenure, 'PhoneService':phone,
    'MultipleLines':multi_lines, 'InternetService':internet, 'OnlineSecurity':online_sec,
    'OnlineBackup':online_bk, 'DeviceProtection':device_prot, 'TechSupport':tech_supp,
    'StreamingTV':streaming_tv, 'StreamingMovies':streaming_m, 'Contract':contracts,
    'PaperlessBilling':paperless, 'PaymentMethod':payment,
    'MonthlyCharges':monthly, 'TotalCharges':total, 'Churn':churn
})

print(f'Dataset shape : {df.shape}')
print(f'Churn rate    : {(df.Churn=="Yes").mean()*100:.1f}%')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head(3)

## 3. Data Cleaning & Feature Engineering

In [ ]:
df_clean = df.copy()

# Drop non-predictive ID
df_clean.drop(columns=['customerID'], inplace=True)

# Fix TotalCharges type
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce').fillna(0)

# Encode target
df_clean['Churn'] = (df_clean['Churn'] == 'Yes').astype(int)

# ── Feature Engineering (same 5 features as the ML project) ─────
# 1. Tenure lifecycle group
df_clean['tenure_group'] = pd.cut(
    df_clean['tenure'], bins=[-1,12,24,48,72],
    labels=['New (0-12m)','Growing (1-2yr)','Established (2-4yr)','Loyal (4yr+)']
)

# 2. Average monthly spend efficiency
df_clean['avg_monthly_spend'] = df_clean['MonthlyCharges'] / (df_clean['tenure'] + 1)

# 3. Bundled services flag
df_clean['has_bundle'] = (
    (df_clean['PhoneService'] == 'Yes') &
    (df_clean['InternetService'].isin(['DSL','Fiber optic']))
).astype(int)

# 4. High-value customer (top 25% by monthly charges)
df_clean['high_value'] = (
    df_clean['MonthlyCharges'] > df_clean['MonthlyCharges'].quantile(0.75)
).astype(int)

# 5. Contract risk score
df_clean['contract_risk'] = df_clean['Contract'].map(
    {'Month-to-month': 2, 'One year': 1, 'Two year': 0}
)

print(f'Shape after feature engineering: {df_clean.shape}')
print(f'\nNew features added: tenure_group, avg_monthly_spend, has_bundle, high_value, contract_risk')
df_clean[['tenure','tenure_group','avg_monthly_spend','has_bundle','high_value','contract_risk']].head(3)

## 4. Preprocessing — Encode & Scale

In [ ]:
y = df_clean['Churn']
X = df_clean.drop(columns=['Churn'])

# Label-encode all categoricals
le = LabelEncoder()
cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()
print(f'Encoding {len(cat_cols)} categorical columns...')
for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))

# Stratified 70/15/15 split  (train / validation / test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Scale numeric features
num_cols = ['tenure','MonthlyCharges','TotalCharges','avg_monthly_spend',
            'SeniorCitizen','has_bundle','high_value','contract_risk']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols]   = scaler.transform(X_val[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

# Convert to numpy arrays for Keras
X_train_arr = X_train.values.astype('float32')
X_val_arr   = X_val.values.astype('float32')
X_test_arr  = X_test.values.astype('float32')
y_train_arr = y_train.values.astype('float32')
y_val_arr   = y_val.values.astype('float32')
y_test_arr  = y_test.values.astype('float32')

print(f'\nSplit summary:')
print(f'  Train      : {X_train_arr.shape}  | Churn rate: {y_train_arr.mean()*100:.1f}%')
print(f'  Validation : {X_val_arr.shape}   | Churn rate: {y_val_arr.mean()*100:.1f}%')
print(f'  Test       : {X_test_arr.shape}   | Churn rate: {y_test_arr.mean()*100:.1f}%')
print(f'  Features   : {X_train_arr.shape[1]}')

## 5. Build the Neural Network

### Architecture:
```
Input (24 features)
  → Dense(64, ReLU) → BatchNorm → Dropout(0.3)
  → Dense(32, ReLU) → BatchNorm → Dropout(0.2)
  → Dense(16, ReLU) → Dropout(0.1)
  → Dense(1, Sigmoid)  ← churn probability
```

**Design choices explained:**
- **ReLU** activation: avoids vanishing gradient, fast to train
- **BatchNormalization**: stabilises training, allows higher learning rates
- **Dropout**: prevents overfitting on small dataset
- **Sigmoid** output: squashes to [0,1] — perfect for binary classification probability
- **Binary crossentropy** loss: standard for binary classification
- **class_weight**: handles class imbalance (25.7% churn)

In [ ]:
def build_model(input_dim):
    model = Sequential([
        # Input
        Input(shape=(input_dim,)),

        # Hidden Layer 1
        Dense(64, activation='relu',
              kernel_regularizer=regularizers.l2(0.001)),
        BatchNormalization(),
        Dropout(0.3),

        # Hidden Layer 2
        Dense(32, activation='relu',
              kernel_regularizer=regularizers.l2(0.001)),
        BatchNormalization(),
        Dropout(0.2),

        # Hidden Layer 3
        Dense(16, activation='relu'),
        Dropout(0.1),

        # Output — sigmoid for binary probability
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall')
        ]
    )
    return model

model = build_model(X_train_arr.shape[1])
model.summary()

## 6. Train the Model

In [ ]:
# Handle class imbalance
neg = (y_train_arr == 0).sum()
pos = (y_train_arr == 1).sum()
class_weight = {0: 1.0, 1: neg / pos}
print(f'Class weights — No Churn: {class_weight[0]:.2f} | Churn: {class_weight[1]:.2f}')

# Callbacks
cb_early_stop = EarlyStopping(
    monitor='val_auc', patience=10,
    restore_best_weights=True, mode='max', verbose=1
)
cb_reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=5, min_lr=1e-6, verbose=1
)

# Train
print('\nTraining neural network...')
history = model.fit(
    X_train_arr, y_train_arr,
    validation_data=(X_val_arr, y_val_arr),
    epochs=100,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[cb_early_stop, cb_reduce_lr],
    verbose=1
)

print(f'\n✅ Training complete! Best epoch: {np.argmax(history.history["val_auc"]) + 1}')

## 7. Training History — Loss & Accuracy Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Neural Network Training History', fontsize=14, fontweight='bold', y=1.02)

TRAIN_C = '#0A7B8C'
VAL_C   = '#E05B5B'

plots = [
    ('loss',     'val_loss',     'Binary Cross-Entropy Loss'),
    ('accuracy', 'val_accuracy', 'Accuracy'),
    ('auc',      'val_auc',      'ROC-AUC'),
]

for ax, (tr_key, val_key, title) in zip(axes, plots):
    epochs = range(1, len(history.history[tr_key]) + 1)
    ax.plot(epochs, history.history[tr_key],  color=TRAIN_C, lw=2,   label='Train')
    ax.plot(epochs, history.history[val_key], color=VAL_C,   lw=2,   label='Validation', linestyle='--')
    best_ep = np.argmax(history.history[val_key]) if 'auc' in val_key or 'accuracy' in val_key \
              else np.argmin(history.history[val_key])
    best_val = history.history[val_key][best_ep]
    ax.axvline(best_ep + 1, color='gray', linestyle=':', lw=1.5, alpha=0.7)
    ax.annotate(f'Best: {best_val:.4f}',
                xy=(best_ep+1, best_val), xytext=(best_ep+3, best_val),
                fontsize=8.5, color='gray')
    ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoch'); ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: training_history.png')

## 8. Evaluate on Test Set

In [ ]:
# Predictions
y_prob = model.predict(X_test_arr, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

# Core metrics
metrics = {
    'Accuracy':  round(accuracy_score(y_test_arr, y_pred),    4),
    'Precision': round(precision_score(y_test_arr, y_pred),   4),
    'Recall':    round(recall_score(y_test_arr, y_pred),      4),
    'F1 Score':  round(f1_score(y_test_arr, y_pred),          4),
    'ROC-AUC':   round(roc_auc_score(y_test_arr, y_prob),     4),
}

print('=' * 45)
print('  NEURAL NETWORK — TEST SET RESULTS')
print('=' * 45)
for k, v in metrics.items():
    print(f'  {k:<12} : {v}')
print('=' * 45)
print(f'\n  Identifies {metrics["Recall"]*100:.1f}% of churners before they leave')

print('\nClassification Report:')
print(classification_report(y_test_arr, y_pred,
                             target_names=['No Churn', 'Churn']))

## 9. Visualise Results — Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Neural Network — Evaluation', fontsize=13, fontweight='bold', y=1.02)

# Confusion Matrix
cm = confusion_matrix(y_test_arr, y_pred)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[0],
            cmap=sns.light_palette('#0A7B8C', as_cmap=True),
            xticklabels=['No Churn','Churn'],
            yticklabels=['No Churn','Churn'],
            linewidths=1, linecolor='white',
            annot_kws={'size':14,'weight':'bold'})
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test_arr, y_prob)
auc = roc_auc_score(y_test_arr, y_prob)
axes[1].plot(fpr, tpr, color='#0A7B8C', lw=2.5,
             label=f'Neural Network (AUC = {auc:.3f})')
axes[1].plot([0,1],[0,1],'--', color='gray', lw=1.5, label='Random Classifier')
axes[1].fill_between(fpr, tpr, alpha=0.08, color='#0A7B8C')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('nn_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: nn_evaluation.png')

## 10. Neural Network vs Classical ML — Comparison

In [ ]:
# Results from the main churn project (classical ML)
comparison = pd.DataFrame([
    {'Model':'Logistic Regression', 'Accuracy':0.6352,'Precision':0.3800,'Recall':0.6584,'F1':0.4819,'ROC-AUC':0.6902},
    {'Model':'Random Forest',       'Accuracy':0.7005,'Precision':0.4135,'Recall':0.3884,'F1':0.4006,'ROC-AUC':0.6802},
    {'Model':'Gradient Boosting',   'Accuracy':0.7544,'Precision':0.5766,'Recall':0.1763,'F1':0.2700,'ROC-AUC':0.6840},
    {'Model':'SVM',                 'Accuracy':0.6175,'Precision':0.3625,'Recall':0.6391,'F1':0.4626,'ROC-AUC':0.6717},
    {'Model':'Neural Network (DNN)',
     'Accuracy': metrics['Accuracy'],
     'Precision':metrics['Precision'],
     'Recall':   metrics['Recall'],
     'F1':       metrics['F1 Score'],
     'ROC-AUC':  metrics['ROC-AUC']},
])

# Style the table
styled = comparison.set_index('Model').style \
    .background_gradient(cmap='Greens', subset=['ROC-AUC','Recall','F1']) \
    .highlight_max(subset=['ROC-AUC','Recall','F1'], color='#c6efce') \
    .format(precision=4) \
    .set_caption('Model Comparison — Classical ML vs Neural Network')

display(styled)

In [ ]:
# Bar chart comparison
metrics_cols = ['Accuracy','Precision','Recall','F1','ROC-AUC']
colors = ['#5B3FD4','#0A7B8C','#E05B5B','#F5A623','#111111']

x      = np.arange(len(metrics_cols))
width  = 0.15
fig, ax = plt.subplots(figsize=(13, 5))

for i, (_, row) in enumerate(comparison.iterrows()):
    vals   = [row[m] for m in metrics_cols]
    offset = (i - len(comparison)/2 + 0.5) * width
    bars   = ax.bar(x + offset, vals, width,
                    label=row['Model'], color=colors[i], alpha=0.88)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=6.5, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(metrics_cols, fontsize=11)
ax.set_ylim(0.55, 1.02); ax.set_ylabel('Score')
ax.set_title('All Models Comparison — Classical ML vs Neural Network',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=8, framealpha=0.9)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('model_comparison_with_nn.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: model_comparison_with_nn.png')

## 11. Save the Model

In [ ]:
# Save in Keras native format
model.save('churn_nn_model.keras')
print('Model saved: churn_nn_model.keras')

# Also save weights only
model.save_weights('churn_nn_weights.h5')
print('Weights saved: churn_nn_weights.h5')

# Verify reload
loaded_model = keras.models.load_model('churn_nn_model.keras')
y_prob_check = loaded_model.predict(X_test_arr[:3], verbose=0).flatten()
print(f'\nModel reload verified. Sample probabilities: {y_prob_check.round(3)}')

## 12. Single Customer Prediction Demo

In [ ]:
def predict_churn(customer_raw: dict) -> dict:
    """
    Predict churn probability for a single customer.
    Uses the trained neural network.
    """
    df_new = pd.DataFrame([customer_raw])

    # Feature engineering
    df_new['tenure_group']       = pd.cut(df_new['tenure'], bins=[-1,12,24,48,72],
                                           labels=['New (0-12m)','Growing (1-2yr)',
                                                   'Established (2-4yr)','Loyal (4yr+)'])
    df_new['avg_monthly_spend']  = df_new['MonthlyCharges'] / (df_new['tenure'] + 1)
    df_new['has_bundle']         = ((df_new['PhoneService']=='Yes') &
                                    (df_new['InternetService'].isin(['DSL','Fiber optic']))).astype(int)
    df_new['high_value']         = (df_new['MonthlyCharges'] > 75).astype(int)
    df_new['contract_risk']      = df_new['Contract'].map({'Month-to-month':2,'One year':1,'Two year':0})

    # Encode
    le2 = LabelEncoder()
    for col in df_new.select_dtypes(include=['object','category']).columns:
        df_new[col] = le2.fit_transform(df_new[col].astype(str))

    # Align columns
    for col in X.columns:
        if col not in df_new.columns:
            df_new[col] = 0
    df_new = df_new[X.columns]

    # Scale
    df_new[num_cols] = scaler.transform(df_new[num_cols])
    arr = df_new.values.astype('float32')

    prob = float(model.predict(arr, verbose=0)[0][0])
    risk = 'HIGH' if prob >= 0.70 else 'MEDIUM' if prob >= 0.40 else 'LOW'

    return {
        'churn_prediction': 'Will Churn' if prob >= 0.5 else 'Will Stay',
        'churn_probability': f'{prob*100:.1f}%',
        'risk_level': risk
    }


# ── Demo predictions ─────────────────────────────────────────
test_customers = [
    {
        'label': 'Profile A — HIGH RISK (new, fiber, month-to-month)',
        'data': {
            'gender':'Male','SeniorCitizen':0,'Partner':'No','Dependents':'No',
            'tenure':2,'PhoneService':'Yes','MultipleLines':'No',
            'InternetService':'Fiber optic','OnlineSecurity':'No','OnlineBackup':'No',
            'DeviceProtection':'No','TechSupport':'No','StreamingTV':'Yes',
            'StreamingMovies':'Yes','Contract':'Month-to-month','PaperlessBilling':'Yes',
            'PaymentMethod':'Electronic check','MonthlyCharges':95.5,'TotalCharges':191.0
        }
    },
    {
        'label': 'Profile B — LOW RISK (loyal, two-year, DSL)',
        'data': {
            'gender':'Female','SeniorCitizen':0,'Partner':'Yes','Dependents':'Yes',
            'tenure':60,'PhoneService':'Yes','MultipleLines':'Yes',
            'InternetService':'DSL','OnlineSecurity':'Yes','OnlineBackup':'Yes',
            'DeviceProtection':'Yes','TechSupport':'Yes','StreamingTV':'No',
            'StreamingMovies':'No','Contract':'Two year','PaperlessBilling':'No',
            'PaymentMethod':'Bank transfer','MonthlyCharges':55.2,'TotalCharges':3312.0
        }
    },
]

print('='*55)
print('  CHURN PREDICTION — NEURAL NETWORK DEMO')
print('='*55)
for c in test_customers:
    result = predict_churn(c['data'])
    print(f"\n{c['label']}")
    print(f"  Prediction   : {result['churn_prediction']}")
    print(f"  Probability  : {result['churn_probability']}")
    print(f"  Risk Level   : {result['risk_level']}")

## 13. Key Takeaways

| Observation | Insight |
|-------------|----------|
| Neural Network vs Logistic Regression | NN generally achieves comparable or better ROC-AUC due to capturing non-linear feature interactions |
| Batch Normalization | Significantly stabilises training on tabular data — prevents internal covariate shift |
| Dropout | Critical for preventing overfitting on small datasets (7k rows) |
| Class Weight | Without class weighting, the model would predict 'No Churn' for everything and get 74% accuracy while being useless |
| EarlyStopping | Prevented overfitting by restoring the best weights before the model started memorising training data |

## 14. What I Would Improve Next

1. **Hyperparameter tuning** — use Keras Tuner (RandomSearch or Hyperband) to optimise layer sizes, dropout rates, and learning rate
2. **SMOTE oversampling** — combine with class weighting for better minority class recall
3. **Streamlit dashboard** — wrap the prediction function in a web UI so business users can input customer data and get real-time churn risk
4. **Cloud deployment** — deploy on AWS Lambda or Google Cloud Run for API access

---

*Built by Sourav Nayak | B.Tech CSE, KIIT University 2026 | [github.com/SOURAV033](https://github.com/SOURAV033)*